# 55 — Generate the SHACL shapes

One shapes graph per schema, matching decision D1. Outputs at repo root:

- `cosmos_bc_v1.shapes.ttl`
- `cosmos_sdtm_v1.shapes.ttl`

**Generated from the published schemas and shipped unmodified.** They are CDISC's
constraints, not this repo's opinion about them. Where the A-Box does not conform,
`60_validate_instances.ipynb` reports the divergence rather than adjusting either
side — see decision D11.

## Configuration

In [ ]:
ROOT      = ".."
DOWNLOADS = "../downloads"
BUILD     = "../build"

SOURCES = {
    "cosmos_bc_v1.shapes.ttl":   f"{BUILD}/cosmos_bc_model.patched.yaml",
    "cosmos_sdtm_v1.shapes.ttl": f"{DOWNLOADS}/cosmos_sdtm_model.yaml",
}

## Generate and canonicalize

Shapes graphs are mostly anonymous nodes — every `sh:property` is one — so the
canonicalization of decision D9 matters more here than anywhere else.

In [ ]:
from pathlib import Path

from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph
from rdflib.compare import isomorphic, to_canonical_graph
from rdflib.namespace import SH

for target, source in SOURCES.items():
    graph = Graph().parse(data=ShaclGenerator(source).serialize(), format="turtle")

    canonical = to_canonical_graph(graph)
    if not isomorphic(canonical, graph) or len(canonical) != len(graph):
        raise RuntimeError(f"{target}: canonicalization changed the graph")

    out = Graph()
    for triple in canonical:
        out.add(triple)
    out.bind("sh", SH)

    turtle = out.serialize(format="turtle")
    again = Graph()
    for triple in to_canonical_graph(Graph().parse(data=turtle, format="turtle")):
        again.add(triple)
    again.bind("sh", SH)
    if again.serialize(format="turtle") != turtle:
        raise RuntimeError(f"{target}: canonical serialization is not stable")

    Path(ROOT, target).write_text(turtle, encoding="utf-8")

    node_shapes = len(set(out.subjects(SH.NodeShape, None))) or len(set(out.subjects(None, SH.NodeShape)))
    print(f"{target:28s} {len(out):>5,} triples  {len(turtle):>7,} chars  stable")

## Confirm

In [ ]:
from rdflib import RDF

for target in SOURCES:
    g = Graph().parse(str(Path(ROOT, target)), format="turtle")
    shapes = set(g.subjects(RDF.type, SH.NodeShape))
    closed = sum(1 for s in shapes if (s, SH.closed, None) in g)
    properties = len(list(g.subject_objects(SH.path)))
    print(f"{target}")
    print(f"    sh:NodeShape     {len(shapes):>5,}   ({closed} closed)")
    print(f"    property shapes  {properties:>5,}")

## Provenance

Generated from the models at the commit pinned in `10_fetch_cosmos.ipynb` — the BC
shapes from the patched copy, so they carry the repaired namespace
(`docs/known-gaps.md` §1a). Nothing in either file is authored.